# Task 2 - CT and MRI fusion
CT is good at sharp edges, MRI is good at soft tissue. Here I combine one CT slice and its matching MRI slice into one image.

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

### Step 1: load both images
Both must be the same size to add them, so I resize the MRI if needed.

In [2]:
ct = cv2.imread('data/ct_slice.png', 0)
mri = cv2.imread('data/mri_slice.png', 0)

if ct.shape != mri.shape:
    mri = cv2.resize(mri, (ct.shape[1], ct.shape[0]))
print(ct.shape, mri.shape)

(512, 512) (512, 512)


### Step 2: equalize both separately
They have very different brightness ranges so I equalize each one on its own before mixing.

In [3]:
ct_eq = cv2.equalizeHist(ct)
mri_eq = cv2.equalizeHist(mri)

plt.figure(figsize=(10,8))
for i, (im, name) in enumerate([(ct,'CT'),(ct_eq,'CT eq'),(mri,'MRI'),(mri_eq,'MRI eq')]):
    plt.subplot(2,2,i+1); plt.imshow(im, cmap='gray', vmin=0, vmax=255); plt.title(name); plt.axis('off')
plt.savefig('output/1_equalization.png')
plt.show()

### Step 3: color map
MRI goes to a JET heatmap. CT stays gray but I convert it to 3 channels so both images have the same shape.

In [4]:
mri_color = cv2.applyColorMap(mri_eq, cv2.COLORMAP_JET)
ct_3ch = cv2.cvtColor(ct_eq, cv2.COLOR_GRAY2BGR)

overlay = cv2.addWeighted(ct_3ch, 0.5, mri_color, 0.5, 0)
cv2.imwrite('output/2_mri_heatmap.png', mri_color)
cv2.imwrite('output/2_overlay.png', overlay)

True

### Step 4: weighted fusion
I tried different alpha (CT) and beta (MRI) values. CT should have the bigger weight to keep the sharp edges, MRI smaller just to show the tissue colors. I chose 0.7 / 0.3.

In [5]:
pairs = [(0.5,0.5), (0.6,0.4), (0.7,0.3), (0.85,0.15)]

plt.figure(figsize=(16,4))
for i, (a, b) in enumerate(pairs):
    f = cv2.addWeighted(ct_3ch, a, mri_color, b, 0)
    plt.subplot(1,4,i+1)
    plt.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    plt.title(f'CT {a} / MRI {b}')
    plt.axis('off')
plt.savefig('output/3_weights.png')
plt.show()

fused = cv2.addWeighted(ct_3ch, 0.7, mri_color, 0.3, 0)
cv2.imwrite('output/3_fused.png', fused)

True

### Step 5: log + gamma on the fused image
After adding, some areas can become too dark or too bright, so I apply log first and then gamma 0.8. I also check how many pixels are completely black or white.

In [6]:
c = 255 / np.log(1 + fused.max())
fused_log = (c * np.log(1 + fused.astype(np.float32))).astype(np.uint8)

table = np.array([255 * (i/255)**0.8 for i in range(256)]).astype(np.uint8)
final = cv2.LUT(fused_log, table)
cv2.imwrite('output/4_final.png', final)

print('black pixels before: %.2f%%  after: %.2f%%' % ((fused==0).mean()*100, (final==0).mean()*100))
print('white pixels before: %.2f%%  after: %.2f%%' % ((fused==255).mean()*100, (final==255).mean()*100))

black pixels before: 13.57%  after: 100.00%
white pixels before: 0.01%  after: 0.00%


/tmp/ipykernel_128/2246774601.py:1: RuntimeWarning: overflow encountered in scalar add
  c = 255 / np.log(1 + fused.max())
/tmp/ipykernel_128/2246774601.py:1: RuntimeWarning: divide by zero encountered in log
  c = 255 / np.log(1 + fused.max())


### Step 6: compare
CT, MRI and fused side by side. I also calculated entropy (how much information) and the average edge strength with Sobel to back it up with numbers.

In [7]:
def entropy(g):
    p = np.bincount(g.ravel(), minlength=256) / g.size
    p = p[p > 0]
    return -(p * np.log2(p)).sum()

def edges(g):
    gx = cv2.Sobel(g, cv2.CV_32F, 1, 0)
    gy = cv2.Sobel(g, cv2.CV_32F, 0, 1)
    return np.hypot(gx, gy).mean()

final_gray = cv2.cvtColor(final, cv2.COLOR_BGR2GRAY)
for name, g in [('CT', ct_eq), ('MRI', mri_eq), ('Fused', final_gray)]:
    print(name, 'entropy =', round(entropy(g), 3), ' edge strength =', round(edges(g), 2))

plt.figure(figsize=(15,5))
plt.subplot(1,3,1); plt.imshow(ct_eq, cmap='gray'); plt.title('CT')
plt.subplot(1,3,2); plt.imshow(cv2.cvtColor(mri_color, cv2.COLOR_BGR2RGB)); plt.title('MRI')
plt.subplot(1,3,3); plt.imshow(cv2.cvtColor(final, cv2.COLOR_BGR2RGB)); plt.title('Fused')
for a in plt.gcf().axes: a.axis('off')
plt.savefig('output/5_comparison.png')
plt.show()

CT entropy = 5.26  edge strength = 107.91
MRI entropy = 6.313  edge strength = 45.95
Fused entropy = -0.0  edge strength = 0.0


### Conclusion
(write in your own words: what the fused image shows that CT/MRI alone do not, and what the numbers say)